# How a phase diagram is built: Mg–Ca

landau finds phase transitions by comparing the free energies of all phases. This notebook shows that in three steps:

- **a)** At 800 K, plot the semigrand potential $\phi$ of each phase against the chemical potential difference $\Delta\mu$. The phase with the lowest $\phi$ is stable, and transitions sit where the curves cross.
- **b)** The same temperature as free energy against concentration. The crossings from a) are now common tangents.
- **c)** Repeat over a range of temperatures to get the full phase diagram. The markers show the 800, 900 and 1000 K transitions.

The free energies in `MgCaFreeEnergies.pckl.gz` come from calphy calculations with an ACE potential.

Saves the figure as `MgCa_Phasediagram_Construction_landau.svg` and `.png`.

## Setup

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import landau as ld

warnings.simplefilter("ignore", DeprecationWarning)

In [ ]:
DATA = "MgCaFreeEnergies.pckl.gz"
OUT = "MgCa_Phasediagram_Construction_landau.svg"

T_SLICE = 800  # temperature of panels a) and b)
T_CUTS = [800, 900, 1000]  # transitions marked in c)
T_RANGE = (700, 1100)
MU_RANGE = (-1.1, -0.1)

MG, MG2CA, CA, LIQ = "hcp Mg", "Mg$_2$Ca", "bcc Ca", "liquid"
PHASE_COLOR = {MG: "#a1c9f4", MG2CA: "#ffb482", CA: "#ff9f9b", LIQ: "#debb9b"}  # fills in c)
LINE_COLOR = {MG: "#4878d0", MG2CA: "#ee854a", CA: "#d65f5f", LIQ: "#8c613c"}  # lines in a) and b)
MIN_C_WIDTH = 0.005  # drawn width of the line phases in c)
CUT_MARKER = {800: "D", 900: "o", 1000: "o"}

FIGSIZE = (10.5, 7.5)
FONTSIZE = 15
DPI = 150

## Phases

The solids are line phases with a fixed composition. The liquid interpolates between free energies at several compositions. `fcc Ca` is not stable in this temperature range, so we leave it out.

In [ ]:
raw = pd.read_pickle(DATA)
raw = raw[raw.phase != "fcc Ca"]


def line(r):
    return ld.phases.TemperatureDependentLinePhase(
        r.phase, r.composition, r.temperature, r.free_energy, interpolator=ld.interpolate.SGTE(4)
    )


solids = [line(r) for _, r in raw[raw.phase != LIQ].iterrows()]
liquid = ld.phases.FastInterpolatingPhase(LIQ, [line(r) for _, r in raw[raw.phase == LIQ].iterrows()])
phases = {p.name: p for p in solids + [liquid]}
phases

## Calculate

One calculation per panel: a fine $\Delta\mu$ scan and a coarser scan at 800 K, then the full temperature range, then the three marked temperatures.

In [ ]:
df_phi = ld.calculate.calc_phase_diagram(
    phases.values(), Ts=[T_SLICE], mu=np.linspace(*MU_RANGE, 200), keep_unstable=True
)
df_f = ld.calculate.calc_phase_diagram(phases.values(), Ts=[T_SLICE], mu=200, keep_unstable=True)
df_full = ld.calculate.calc_phase_diagram(phases.values(), Ts=np.linspace(*T_RANGE, 50), mu=200)
df_cuts = pd.concat(
    [ld.calculate.calc_phase_diagram(phases.values(), Ts=[T], mu=200) for T in T_CUTS], ignore_index=True
)

In [ ]:
def transitions(df, T):
    """Coexisting phase rows at exactly temperature T, one group per mu."""
    b = df.query("locus == 'boundary' and T == @T").sort_values(["mu", "c"])
    return [g for _, g in b.groupby("mu")]


pd.concat([g for T in T_CUTS for g in transitions(df_cuts, T)])[["T", "phase", "mu", "c"]]

## Plot helpers

In [ ]:
def _plain(label):
    """Phase label without bold math, so it can be compared to the phase name."""
    return re.sub(r"\\mathbf\{([^}]*)\}", r"\1", label)


def phase_texts(ax, names):
    return [t for t in ax.texts if _plain(t.get_text()) in names]


def rotate_phase_labels(ax, names):
    """Turn the phase labels on ax vertical."""
    for t in phase_texts(ax, names):
        t.set(rotation=90, rotation_mode="anchor")

## Panels

In [ ]:
def panel_phi(ax, df):
    cycle = plt.cycler(color=[LINE_COLOR[n] for n in sorted(phases)])
    with plt.rc_context({"axes.prop_cycle": cycle, "lines.linewidth": 2}):
        ld.plot.plot_1d_mu_phase_diagram(df, ax=ax, ylim=(-2.1, -1.6), mark_transitions=False, element="Ca")
    for g in transitions(df, T_SLICE):
        ax.scatter(g.mu.iloc[0], g.phi.iloc[0], marker="D", s=45, color="k", edgecolor="w", lw=0.8, zorder=5)
    for t in phase_texts(ax, phases):
        t.set(rotation=90, rotation_mode="anchor", ha="right", va="center")
    ax.set_xlim(*MU_RANGE)
    ax.set_ylabel(r"$\phi = f - \Delta\mu\,c$ [eV/atom]")
    ax.set_title(f"T = {T_SLICE} K")


def panel_f(ax, df):
    plt.sca(ax)
    ld.plot.plot_excess_free_energy(df, color_override=LINE_COLOR, element="Ca")
    rotate_phase_labels(ax, phases)
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylabel(r"$f_\mathrm{excess}$ [eV/atom]")
    ax.set_title(f"T = {T_SLICE} K")


def panel_diagram(ax, df_full, df_cuts):
    ld.plot.plot_phase_diagram(
        df_full, element="Ca", color_override=PHASE_COLOR, legend=False, triplepoints=True, transition_temperatures=True, ax=ax,
        poly_method=ld.poly.SegmentFastTsp(min_c_width=MIN_C_WIDTH),
    )
    for T in T_CUTS:
        for g in transitions(df_cuts, T):
            for _, r in g.iterrows():
                ax.scatter(r.c, T, marker=CUT_MARKER[T], s=45, color=PHASE_COLOR[r.phase],
                           edgecolor="k", lw=0.8, zorder=10, clip_on=False)
    ax.set_xlim(0, 1)
    ax.set_ylim(*T_RANGE)
    # labels beside the narrow line phases
    for x, ha, n in [(0.012, "left", MG), (0.322, "right", MG2CA), (0.988, "right", CA)]:
        ax.text(x, 733, n, rotation=90, ha=ha, va="bottom", fontsize=FONTSIZE - 2)
    ax.text(0.65, 1000, "liquid", ha="center", va="center")

## Figure

In [ ]:
plt.style.use("default")
plt.rcParams.update({"font.size": FONTSIZE, "svg.fonttype": "none"})
fig = plt.figure(figsize=FIGSIZE, layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=[1, 0.95])
ax_a, ax_b = top.subplots(1, 2)
ax_c = bottom.subplots()

panel_phi(ax_a, df_phi)
panel_f(ax_b, df_f)
panel_diagram(ax_c, df_full, df_cuts)

for ax, lab in zip((ax_a, ax_b, ax_c), "abc"):
    ax.annotate(f"{lab})", xy=(0, 1), xycoords="axes fraction", xytext=(-50, 4), textcoords="offset points",
                ha="right", va="bottom", fontsize=FONTSIZE + 1.5)

fig.savefig(OUT)
fig.savefig(OUT.replace(".svg", ".png"), dpi=DPI, transparent=True)